In [ ]:
import cv2
import torch
import numpy as np
import time
import threading
import winsound  # WINDOWS ONLY
from ultralytics import YOLO
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from PIL import Image

# --- CONFIGURATION ---
VIDEO_PATH = "long_street_clip.mp4"    # Replace with your video path
MODEL_PATH = "bdd100k_7model.pt"   # Your NEW 7-class model file

# --- BEEP LOGIC ---
current_danger_level = 0 

def beep_loop():
    while True:
        if current_danger_level == 0:
            time.sleep(0.1)
        elif current_danger_level == 1: # FAR (5-10m)
            winsound.Beep(600, 200) 
            time.sleep(1.0) 
        elif current_danger_level == 2: # CLOSE (2-5m)
            winsound.Beep(900, 150)
            time.sleep(0.3)
        elif current_danger_level == 3: # CRITICAL (<2m)
            winsound.Beep(1500, 100)
            time.sleep(0.05) 

audio_thread = threading.Thread(target=beep_loop, daemon=True)
audio_thread.start()

# MODEL SETUP
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"System: Using {device.upper()}")

model = YOLO(MODEL_PATH)
# 7 Classes from the dataset
class_names = {0: 'bike', 1: 'bus', 2: 'car', 3: 'motor', 4: 'person', 5: 'rider', 6: 'truck'}

depth_id = "depth-anything/Depth-Anything-V2-Metric-Outdoor-Small-hf"
image_processor = AutoImageProcessor.from_pretrained(depth_id)
depth_model = AutoModelForDepthEstimation.from_pretrained(depth_id).to(device)

# PROCESSING
cap = cv2.VideoCapture(VIDEO_PATH)
width, height = int(cap.get(3)), int(cap.get(4))
out = cv2.VideoWriter("7class_demo.mp4", cv2.VideoWriter_fourcc(*'mp4v'), 30, (width, height))

print("Processing 7-Class Model... Press 'Q' to quit.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # 1. Depth
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    inputs = image_processor(images=Image.fromarray(image_rgb), return_tensors="pt").to(device)
    with torch.no_grad():
        depth_map = depth_model(**inputs).predicted_depth
        depth_map = torch.nn.functional.interpolate(
            depth_map.unsqueeze(1), size=(height, width), mode="bicubic", align_corners=False
        ).squeeze().cpu().numpy()

    # 2. YOLO
    # Lower confidence to 0.25 to catch more pedestrians/bikes
    results = model(frame, conf=0.25, verbose=False)[0]

    # 3. Logic
    min_dist = 999.0
    
    for box in results.boxes:
        coords = box.xyxy.cpu().numpy()[0]
        x1, y1, x2, y2 = map(int, coords)
        cls_id = int(box.cls[0])
        
        # Safety Check: If model predicts a class ID outside our 7 classes, skip it
        if cls_id not in class_names: continue
        label = class_names[cls_id]

        # FILTER: Beep for everything in the list (Bike, Car, Person, Bus, Truck)
        # We assume 'rider' and 'motor' are also dangerous
        
        # Depth Calc
        h_img, w_img = depth_map.shape
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w_img, x2), min(h_img, y2)
        
        obj_depth = depth_map[y1:y2, x1:x2]
        if obj_depth.size > 0:
            dist = np.median(obj_depth)
            
            if dist < min_dist:
                min_dist = dist

            # Visuals
            color = (0, 255, 0)
            if dist < 5: color = (0, 0, 255)
            
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"{label} {dist:.1f}m", (x1, y1-10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # 4. Danger Level
    if min_dist < 2.0:
        current_danger_level = 3
        cv2.putText(frame, "CRITICAL!", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 2, (0,0,255), 3)
    elif min_dist < 5.0:
        current_danger_level = 2
    elif min_dist < 10.0:
        current_danger_level = 1
    else:
        current_danger_level = 0

    out.write(frame)
    cv2.imshow("7-Class Demo", frame)
    if cv2.waitKey(1) == ord('q'): break

cap.release()
out.release()
cv2.destroyAllWindows()

System: Using CUDA
Processing 7-Class Model... Press 'Q' to quit.


## MODEL VALIDATION

> **Note (repository):** the output below was produced on the original Windows laptop (Ultralytics 8.3.232, Quadro P1000) and is kept unchanged. Its `mAP50 = 0.517` could not be reproduced. Re-validating with default settings gives **mAP@50 = 0.430** (see README); with `conf=0.25, iou=0.6` it gives P 0.596, R 0.391, mAP@50 0.343. Download the dataset into `bdd100k_7class/` before running.


In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

# Load Model
model = YOLO('bdd100k_7model.pt')

# Run Validation
# This automatically generates Confusion Matrix & PR Curves
print("Running Validation... This may take a few minutes.")
metrics = model.val(
    data="bdd100k_7class/data.yaml",  # Point to your dataset config
    split='val',       # Use validation set
    conf=0.25,         # Confidence threshold
    iou=0.6,
    plots=True         # CRITICAL: This tells YOLO to save the charts
)

print(f" Charts saved in: {metrics.save_dir}")
print("Go there to find 'confusion_matrix_normalized.png' and 'PR_curve.png'")

Running Validation... This may take a few minutes.
Ultralytics 8.3.232  Python-3.10.19 torch-2.7.1+cu118 CUDA:0 (Quadro P1000, 4096MiB)
Model summary (fused): 72 layers, 3,007,013 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 4.92.3 MB/s, size: 43.2 KB)
val: Scanning C:\Users\USER\Desktop\TUM\Big Geospatial Data\walking_stick_project\bdd100k_7class\valid\labels... 1960 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1960/1960 203.9it/s 9.6s<0.0s
val: New cache created: C:\Users\USER\Desktop\TUM\Big Geospatial Data\walking_stick_project\bdd100k_7class\valid\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 123/123 2.9it/s 42.3s0.3s
                   all       1960      24550      0.595       0.39      0.517      0.331
                  bike        121        221      0.546      0.294      0.449      0.233
                   bus        227        309      0.557      0.411    

## DEPTHANYTHING EXPORT TO ONYX

In [27]:
import torch
from transformers import AutoModelForDepthEstimation

# 1. Load Model
print("Loading DepthAnything Model...")
model = AutoModelForDepthEstimation.from_pretrained("depth-anything/Depth-Anything-V2-Metric-Outdoor-Small-hf")
model.eval()

# 2. Create Dummy Input
dummy_input = torch.randn(1, 3, 518, 518)

# 3. Export with OPSET 14 (The Fix)
print("Exporting to ONNX...")
torch.onnx.export(
    model, 
    dummy_input, 
    "depth_model.onnx", 
    opset_version=14,       # <--- CHANGED FROM 11 TO 14
    input_names=["pixel_values"], 
    output_names=["predicted_depth"]
)

print("Success! Created depth_model.onnx")

Loading DepthAnything Model...
Exporting to ONNX...


C:\Users\USER\anaconda3\envs\vision_project\lib\site-packages\transformers\models\dinov2\modeling_dinov2.py:143: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if num_channels != self.num_channels:
C:\Users\USER\anaconda3\envs\vision_project\lib\site-packages\transformers\models\depth_anything\modeling_depth_anything.py:159: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if hidden_state.shape != residual.shape:
C:\Users\USER\anaconda3\envs\vision_project\lib\site-packages\transformers\models\depth_anything\modeling_depth_anything.py:312: TracerWarning: C

Success! Created depth_model.onnx


## sound generation

In [32]:
import wave
import math
import struct

def create_beep(filename, frequency=1000, duration_ms=200):
    # Audio configuration
    sample_rate = 44100
    n_samples = int(sample_rate * (duration_ms / 1000.0))
    amplitude = 16000  # Volume (Max 32767)
    
    with wave.open(filename, 'w') as wav_file:
        # Set parameters: 1 channel (mono), 2 bytes size, 44100 Hz
        wav_file.setparams((1, 2, sample_rate, n_samples, 'NONE', 'not compressed'))
        
        # Write the sound wave
        for i in range(n_samples):
            # Math to create a Sine Wave sound
            value = int(amplitude * math.sin(2 * math.pi * frequency * i / sample_rate))
            # Pack as binary data
            data = struct.pack('<h', value)
            wav_file.writeframes(data)
            
    print(f" Generated: {filename} ({frequency}Hz, {duration_ms}ms)")

# 1. Create the "Fast/Critical" Beep (High pitch, short)
create_beep("beep_fast.wav", frequency=1500, duration_ms=100)

# 2. Create the "Slow/Warning" Beep (Lower pitch, longer)
create_beep("beep_slow.wav", frequency=600, duration_ms=300)

 Generated: beep_fast.wav (1500Hz, 100ms)
 Generated: beep_slow.wav (600Hz, 300ms)
